# Semantic Search Engine Interactive Activity
*AIPI 510: Data Sourcing for Analytics*

*Week 13*



---



**Build Your Own Semantic Search Engine**

> **Goal**: Create a semantic search app from
scratch and deploy it to Hugging Face Spaces.


1.   Pick a dataset
2.  Chunk + embed your data
3. Store your data in a Chroma vector db
4. Build a search app using Streamlit or Gradio
5. Publish on Hugging Face Spaces
6. Submit Link on Canvas



---

*This is a starter notebook. This is designed to give you a jumping off point only. *

Make sure to
`pip install chromadb gradio sentence-transformers`


In [2]:
%pip install "scipy>=1.17.1" sentence-transformers chromadb streamlit datasets ipywidgets

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 4.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 27.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 30.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.3/608.3 kB 21.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 12.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 29.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 32.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 27.5 MB/s eta 0:00:00
  Using cached tqdm-4.70.1-py3-none-any.whl (80 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.1/494.1 kB 22.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 13.0 MB/s eta 0:00:00
     ━━━━━━━━━

In [1]:
# Load an embedding model (you can use any embedding model)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

W1117 13:00:05.602000 30894 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [2]:
from datasets import load_dataset

# 1. Load the SciQ training dataset
dataset = load_dataset("allenai/sciq", split="train")

# 2. Remove records without supporting explanations
dataset = dataset.filter(
    lambda row: bool((row["support"] or "").strip())
)

# 3. Select 1,000 records for this project
dataset = dataset.select(range(min(1000, len(dataset))))

# 4. Combine each question, correct answer, and explanation
texts = [
    f"Question: {row['question']}\n"
    f"Correct answer: {row['correct_answer']}\n"
    f"Explanation: {row['support']}"
    for row in dataset
]

# 5. Preview the data
print(f"Loaded {len(texts)} science records.")
print("\nFirst record:\n")
print(texts[0])

README.md:   0%|          | 0.00/7.02k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.99MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  339kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  343kB            

... (2 outputs left out)

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11679 [00:00<?, ? examples/s]

Loaded 1000 science records.

First record:

Question: What type of organism is commonly used in preparation of foods such as cheese and yogurt?
Correct answer: mesophilic organisms
Explanation: Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many pathogenic mesophiles is 37°C (98°F), the normal human body temperature. Mesophilic organisms have important uses in food preparation, including cheese, yogurt, beer and wine.


In [3]:
# Split text into overlapping chunks using the model's tokenizer
def chunk_text(text, chunk_size=200, overlap=30):
    token_ids = model.tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []
    start = 0

    while start < len(token_ids):
        end = min(start + chunk_size, len(token_ids))

        chunk = model.tokenizer.decode(
            token_ids[start:end],
            skip_special_tokens=True
        )
        chunks.append(chunk)

        if end == len(token_ids):
            break

        start = end - overlap

    return chunks


# Keep each chunk linked to its original science record
chunks = []
metadatas = []
chunk_ids = []

for record_id, text in enumerate(texts):
    record_chunks = chunk_text(text)

    for chunk_index, chunk in enumerate(record_chunks):
        chunks.append(chunk)

        metadatas.append({
            "record_id": record_id,
            "chunk_index": chunk_index,
            "question": dataset[record_id]["question"],
            "correct_answer": dataset[record_id]["correct_answer"]
        })

        chunk_ids.append(f"record_{record_id}_chunk_{chunk_index}")

print(f"Original records: {len(texts)}")
print(f"Total chunks: {len(chunks)}")
print("\nFirst chunk:\n")
print(chunks[0])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (382 > 256). Running this sequence through the model will result in indexing errors


Original records: 1000
Total chunks: 1201

First chunk:

question : what type of organism is commonly used in preparation of foods such as cheese and yogurt? correct answer : mesophilic organisms explanation : mesophiles grow best in moderate temperature, typically between 25°c and 40°c ( 77°f and 104°f ). mesophiles are often found living in or on the bodies of humans or other animals. the optimal growth temperature of many pathogenic mesophiles is 37°c ( 98°f ), the normal human body temperature. mesophilic organisms have important uses in food preparation, including cheese, yogurt, beer and wine.


In [4]:
# Generate embeddings for the chunks
embeddings = model.encode(
    chunks,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Embedding shape: (1201, 384)


In [5]:
# Store embeddings in ChromaDB
import chromadb

# Save the database in the project's chroma_db folder
client = chromadb.PersistentClient(path="./chroma_db")

# Create or reuse a collection with cosine distance
collection = client.get_or_create_collection(
    name="sciq_chunks",
    configuration={"hnsw": {"space": "cosine"}},
    embedding_function=None
)

# Store chunks, embeddings, and source information
collection.upsert(
    ids=chunk_ids,
    documents=chunks,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print(f"Stored {collection.count()} chunks in Chroma.")

Stored 1201 chunks in Chroma.


In [6]:
# 1. Enter a science question
query = "How do plants get energy from sunlight?"

# 2. Embed the query using the same model
query_embedding = model.encode(
    [query],
    normalize_embeddings=True
)

# 3. Retrieve similar chunks
results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=min(20, collection.count()),
    include=["documents", "metadatas", "distances"]
)

# 4. Show up to 5 different science records
seen_records = set()

print(f"Search: {query}\n")

for metadata, distance in zip(
    results["metadatas"][0],
    results["distances"][0]
):
    record_id = metadata["record_id"]

    # Avoid showing the same record more than once
    if record_id in seen_records:
        continue

    seen_records.add(record_id)
    record = dataset[record_id]

    print(f"Result {len(seen_records)}")
    print("Question:", record["question"])
    print("Answer:", record["correct_answer"])
    print("Explanation:", record["support"])
    print(f"Cosine similarity: {1 - distance:.3f}")
    print("-" * 70)

    if len(seen_records) == 5:
        break

Search: How do plants get energy from sunlight?

Result 1
Question: During photosynthesis, what is energy from the sun converted to after entering a plant?
Answer: glucose
Explanation: Cellular respiration and photosynthesis are direct opposite reactions. Energy from the sun enters a plant and is converted into glucose during photosynthesis. Some of the energy is used to make ATP in the mitochondria during cellular respiration, and some is lost to the environment as heat.
Cosine similarity: 0.695
----------------------------------------------------------------------
Result 2
Question: Millions of years ago, plants used energy from the sun to form what?
Answer: carbon compounds
Explanation: Fossil fuels include coal, oil, and natural gas. Fossil fuels are the greatest energy source for modern society. Millions of years ago, plants used energy from the Sun to form carbon compounds. These compounds were later transformed into coal, oil, or natural gas. Fossil fuels take millions of years 

In [ ]:
# Define a search function
def semantic_search(query):
    query_embedding = model.encode([query])
    results = collection.query(query_embeddings=query_embedding.tolist(), n_results=1)
    return "\n\n".join(results['documents'][0])

In [ ]:
# Simple Gradio interface
import gradio as gr
demo = gr.Interface(
    fn=semantic_search,
    inputs=gr.Textbox(label="Enter your search query"),
    outputs=gr.Textbox(label="Top Matches"),
    title="Semantic Search Engine",
    description="Search over your custom dataset using semantic similarity."
)

demo.launch()

### Next Steps:

- Save your chromadb
- Write code to load your chromadb
- Convert your loading, search function and app code to an app.py script
- Create a requirements.txt file
- Test your code!!
- Push your git repo to Hugging Face Spaces


___
Your file tree in your repository will look like this:

- chromadb/
  - chroma.sqlite3
- README.md
- requirements.txt
- app.py